# MERIT-UNETR++ training on Kaggle

Select **GPU T4 x2** or **GPU P100** in Kaggle Settings, then run cells in order.
This uses fold 0, 11 index slices, 256x256, batch size 1 and native scaled intensity.

**Prerequisites:**
1. Upload `data/covid19_20/` (cases.json + Train/) as a Kaggle Dataset named e.g. `merit-covid19-20`
2. Add that dataset to this notebook via **Add Data** on the right panel
3. The dataset will appear at `/kaggle/input/<dataset-name>/`

Checkpoints are saved to `/kaggle/working/` (persists as notebook output).
The three cases with missing spatial units are explicitly assumed to be mm.
The test partition is not evaluated by this notebook.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, subprocess, sys, tempfile

# ===== CONFIGURE THESE =====
REPO_URL = 'https://github.com/torusama/merit-covid-ct.git'
BRANCH = 'train/merit-unetrpp-colab'
PROJECT = Path('/kaggle/working/merit-covid-train')

# Change this to match YOUR Kaggle dataset name.
# After uploading, the data appears at /kaggle/input/<dataset-slug>/
# If your dataset slug is 'merit-covid19-20', the path is:
KAGGLE_DATASET_SLUG = 'merit-covid19-20'  # <-- CHANGE THIS to your actual dataset slug

# Try common layouts: data might be at root or inside a subfolder
_input_base = Path(f'/kaggle/input/{KAGGLE_DATASET_SLUG}')
if (_input_base / 'cases.json').is_file():
    DATA = _input_base
elif (_input_base / 'covid19_20' / 'cases.json').is_file():
    DATA = _input_base / 'covid19_20'
elif (_input_base / 'data' / 'covid19_20' / 'cases.json').is_file():
    DATA = _input_base / 'data' / 'covid19_20'
else:
    # List what's actually there to help debug
    print('Contents of', _input_base, ':')
    if _input_base.exists():
        for p in sorted(_input_base.rglob('*'))[:30]:
            print(' ', p.relative_to(_input_base))
    else:
        print('  (directory does not exist)')
    raise FileNotFoundError(
        f'Cannot find cases.json under {_input_base}. '
        f'Check KAGGLE_DATASET_SLUG and your upload structure.'
    )

RUN_ROOT = Path('/kaggle/working/merit_runs')
RUN_NAME = 'fold0_256_native_v1'
FOLD = 0
EPOCHS = 1  # Total epoch budget; increase to continue a successful run.
CONFIG = PROJECT / 'configs/merit_colab_256.yaml'

print(f'Data path: {DATA}')
print(f'Project path: {PROJECT}')

def run_command(args, cwd=None):
    process = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    if code:
        raise subprocess.CalledProcessError(code, args)

## 1. Clone the training branch from GitHub
The branch must be pushed to GitHub first. On re-run it pulls new commits if the checkout is clean.

In [ ]:
if not PROJECT.exists():
    run_command(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT)])
else:
    if not (PROJECT / '.git').is_dir():
        raise RuntimeError(f'{PROJECT} exists but is not a Git checkout')
    def git_output(*args):
        return subprocess.check_output(['git', *args], cwd=PROJECT, text=True).strip()
    if git_output('remote', 'get-url', 'origin') != REPO_URL or git_output('branch', '--show-current') != BRANCH:
        raise RuntimeError('Existing checkout has a different origin or branch')
    if git_output('status', '--porcelain'):
        raise RuntimeError('Existing checkout has changes; preserve them before updating')
    run_command(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=PROJECT)
os.chdir(PROJECT)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
print('Git commit:', commit)
if not CONFIG.is_file():
    raise FileNotFoundError(f'Training config missing: {CONFIG}')

## 2. Check dataset and link it into the project
Kaggle datasets are read-only at `/kaggle/input/`. We symlink them into the project tree.

In [ ]:
manifest_path = DATA / 'cases.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Manifest not found: {manifest_path}')
records = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
if isinstance(records, dict):
    records = records['cases']
missing = [str((DATA / r[k]).relative_to(DATA)) for r in records
           for k in ('image', 'mask' if 'mask' in r else 'infection_mask') if not (DATA / r[k]).is_file()]
print(f'Cases: {len(records)}, missing CT/mask files: {len(missing)}')
if len(records) != 199 or missing:
    print('Missing:', *missing, sep='\n  ')
    raise RuntimeError('Complete the dataset upload before continuing')
link = PROJECT / 'data/covid19_20'
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink():
    if link.resolve() != DATA.resolve():
        raise RuntimeError('Existing data link points elsewhere')
elif link.exists():
    raise RuntimeError(f'Existing dataset path will not be overwritten: {link}')
else:
    link.symlink_to(DATA, target_is_directory=True)
print('Dataset linked:', link, '->', link.resolve())

## 3. Install dependencies and verify
Installation runs once per Kaggle session.

In [ ]:
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[unetrpp]'], cwd=PROJECT)
run_command([sys.executable, 'scripts/merit_pipeline.py', 'check-split', '--config', str(CONFIG)], cwd=PROJECT)
run_command([sys.executable, '-c',
    "import torch; print('CUDA:', torch.cuda.is_available()); "
    "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); "
    "assert torch.cuda.is_available(), 'Select a GPU kernel in Kaggle Settings'"
], cwd=PROJECT)

## 4. Required real-CT pilot
Four actual training steps on a real CT volume, then checkpoint reload and inference.
The training cell requires this pilot to pass.

In [ ]:
pilot_report = None
pilot_gate = None
pilot_output = Path(tempfile.mkdtemp(prefix='merit_pilot_')) / 'run'
run_command([sys.executable, '-u', 'scripts/merit_preflight.py', 'pilot', '--config', str(CONFIG),
    '--fold', str(FOLD), '--case-id', 'volume-covid19-A-0025', '--size', '256',
    '--steps', '4', '--device', 'cuda', '--output', str(pilot_output)], cwd=PROJECT)
pilot_report = json.loads((pilot_output / 'pilot.json').read_text(encoding='utf-8'))
if pilot_report.get('status') != 'PASS':
    raise RuntimeError(f'Pilot failed: {pilot_report.get("error", "unknown failure")}')
pilot_gate = (commit, hashlib.sha256(CONFIG.read_bytes()).hexdigest(), FOLD, str(DATA.resolve()))
print('Real CT pilot: PASS; output:', pilot_output)

## 5. Train / resume one fold
The initial budget is one **full** training epoch plus validation, which may take hours.
Set `EPOCHS` to a larger total to continue. Checkpoints save to `/kaggle/working/merit_runs/`.

**Important for Kaggle:** After the session ends, download `last.pt` and `best.pt` from
the Output tab, or save the notebook version. To resume in a new session, upload the
checkpoint as a Kaggle Dataset and adjust the resume path below.

In [ ]:
if not isinstance(globals().get('pilot_report'), dict) or pilot_report.get('status') != 'PASS':
    raise RuntimeError('Run the real-CT pilot successfully before training')
current_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT, text=True).strip()
if dirty or current_commit != commit:
    raise RuntimeError('Code changed after setup; use a clean checkout and rerun the pilot')
expected_gate = (current_commit, hashlib.sha256(CONFIG.read_bytes()).hexdigest(), FOLD, str(DATA.resolve()))
if globals().get('pilot_gate') != expected_gate:
    raise RuntimeError('Configuration, fold or data path changed after the pilot; rerun it')
if type(EPOCHS) is not int or EPOCHS < 1 or FOLD not in range(5):
    raise ValueError('EPOCHS must be positive and FOLD must be 0..4')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', RUN_NAME) or RUN_NAME in ('.', '..'):
    raise ValueError('Choose a simple run name')
run_dir = RUN_ROOT / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
record_path = run_dir / 'run_provenance.json'
record = {'git_commit': commit, 'config_sha256': hashlib.sha256(CONFIG.read_bytes()).hexdigest(),
          'dataset': str(DATA.resolve()), 'fold': FOLD, 'training_order': 'case_grouped_v1'}
if record_path.exists():
    if json.loads(record_path.read_text(encoding='utf-8')) != record:
        raise RuntimeError('Existing run has different code, config, data path or fold; choose a new RUN_NAME')
else:
    if any(run_dir.iterdir()):
        raise RuntimeError('Output directory is not empty; choose a new RUN_NAME')
    with record_path.open('x', encoding='utf-8') as handle:
        json.dump(record, handle, indent=2)
checkpoint = run_dir / 'last.pt'

# === RESUME FROM UPLOADED CHECKPOINT ===
# If resuming from a previous Kaggle session, upload last.pt as a dataset
# and set the path here:
# RESUME_FROM_INPUT = Path('/kaggle/input/<your-checkpoint-dataset>/last.pt')
RESUME_FROM_INPUT = None  # Set to a Path to resume from an uploaded checkpoint

args = [sys.executable, '-u', 'scripts/train_colab.py', '--config', str(CONFIG),
        '--fold', str(FOLD), '--output', str(run_dir), '--epochs', str(EPOCHS)]
if checkpoint.is_file():
    print('Resuming from working dir:', checkpoint)
    args += ['--resume', str(checkpoint)]
elif RESUME_FROM_INPUT is not None and Path(RESUME_FROM_INPUT).is_file():
    # Copy uploaded checkpoint to working dir first
    import shutil
    shutil.copy2(RESUME_FROM_INPUT, checkpoint)
    print('Resuming from uploaded checkpoint:', RESUME_FROM_INPUT)
    args += ['--resume', str(checkpoint)]
elif any(p.name != record_path.name for p in run_dir.iterdir()):
    raise RuntimeError('Run contains files but has no last.pt; inspect it or choose a new RUN_NAME')
print('Training output:', run_dir)
run_command(args, cwd=PROJECT)

## 6. Save checkpoints
Kaggle keeps `/kaggle/working/` as notebook output. After training completes or
the session ends, go to the **Output** tab to download `last.pt` and `best.pt`.

To resume in a new session:
1. Upload `last.pt` as a new Kaggle Dataset (e.g. `merit-checkpoint-v1`)
2. Add it to this notebook via **Add Data**
3. Set `RESUME_FROM_INPUT = Path('/kaggle/input/merit-checkpoint-v1/last.pt')` in cell 5
4. Increase `EPOCHS` and re-run all cells

In [ ]:
# List saved files
run_dir = RUN_ROOT / RUN_NAME
if run_dir.exists():
    print('Saved files in', run_dir, ':')
    for f in sorted(run_dir.iterdir()):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'  {f.name}  ({size_mb:.1f} MB)')
else:
    print('No run directory found yet.')

Model checkpoints are research artifacts, not clinically validated results.